Model ini terdiri dari dua bagian:

  1. estimate_price(date)
     Fungsi bantu yang mengestimasi harga pasar gas alam pada TANGGAL BERAPA PUN
     (baik di dalam rentang data historis maupun di masa depan/masa lalu di luar
     rentang data) menggunakan data bulanan di Nat_Gas.csv. Di dalam rentang data,
     dipakai interpolasi linear antar titik bulanan. Di luar rentang data, dipakai
     ekstrapolasi tren + pola musiman (seasonal) yang di-fit dari data historis.

  2. price_storage_contract(...)
     Fungsi utama untuk MENETAPKAN HARGA (value) sebuah kontrak penyimpanan gas.
     Fungsi ini menerima jadwal injeksi (pembelian & penyimpanan) dan penarikan
     (penjualan) gas, lalu menghitung seluruh arus kas: pembelian, penjualan,
     biaya sewa penyimpanan (storage fee), biaya injeksi/penarikan, dan biaya
     transportasi -- lalu mengembalikan nilai bersih kontrak.

Asumsi (sesuai instruksi tugas):
  - Tidak ada keterlambatan transportasi (gas langsung tersedia begitu dibeli/dijual).
  - Suku bunga = 0 (tidak ada diskonto nilai waktu uang).
  - Hari libur pasar/akhir pekan/hari libur bank diabaikan.
  - Tanggal penyuntikan HARUS selalu lebih awal dari tanggal penarikan yang
    memakai volume tersebut (first-in-first-out atas volume yang tersimpan).




In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## BAGIAN 1: MODEL ESTIMASI HARGA DARI DATA HISTORIS (interpolasi + ekstrapolasi)


In [ ]:
def load_price_data(csv_path="/content/drive/MyDrive/Forage/Nat_Gas.csv"):
    """Memuat data harga historis dan menyiapkannya untuk interpolasi/ekstrapolasi."""
    df = pd.read_csv(csv_path)
    df["Dates"] = pd.to_datetime(df["Dates"], format="%m/%d/%y")
    df = df.sort_values("Dates").reset_index(drop=True)
    # Nilai numerik hari (ordinal) dipakai sebagai variabel X untuk regresi tren.
    df["ordinal"] = df["Dates"].map(datetime.toordinal)
    return df


def _fit_trend_and_seasonality(df):
    """
    Fit tren linear jangka panjang + komponen musiman bulanan (12 bulan) dari
    residual terhadap tren. Dipakai untuk mengekstrapolasi harga di luar rentang
    data historis (misalnya musim dingin tahun depan).
    """
    x = df["ordinal"].values.astype(float)
    y = df["Prices"].values.astype(float)

    # Tren linear: price ~ a * ordinal + b
    a, b = np.polyfit(x, y, 1)
    trend = a * x + b
    residual = y - trend

    # Rata-rata residual musiman per bulan (pola musiman berulang tiap tahun)
    months = df["Dates"].dt.month.values
    seasonal_by_month = {}
    for m in range(1, 13):
        mask = months == m
        seasonal_by_month[m] = residual[mask].mean() if mask.any() else 0.0

    return a, b, seasonal_by_month


def estimate_price(date, df=None, csv_path="Nat_Gas.csv"):
    """
    Mengestimasi harga gas alam ($/MMBtu) pada `date` (str 'YYYY-MM-DD' atau
    datetime/Timestamp).

    - Jika `date` berada di dalam rentang data historis -> interpolasi linear.
    - Jika `date` berada di luar rentang (masa depan/masa lalu) -> ekstrapolasi
      memakai tren linear + penyesuaian musiman bulanan.
    """
    if df is None:
        df = load_price_data(csv_path)

    date = pd.to_datetime(date)
    min_date, max_date = df["Dates"].min(), df["Dates"].max()

    if min_date <= date <= max_date:
        # Interpolasi linear langsung terhadap ordinal tanggal
        x_target = date.toordinal()
        price = np.interp(x_target, df["ordinal"], df["Prices"])
        return float(price)
    else:
        # Ekstrapolasi: tren + musiman
        a, b, seasonal_by_month = _fit_trend_and_seasonality(df)
        x_target = date.toordinal()
        trend_value = a * x_target + b
        seasonal_adj = seasonal_by_month.get(date.month, 0.0)
        return float(trend_value + seasonal_adj)

## BAGIAN 2: MODEL PENETAPAN HARGA KONTRAK PENYIMPANAN



In [ ]:
def price_storage_contract(
    injection_dates,
    withdrawal_dates,
    injection_withdrawal_rate,   # MMBtu maksimum yang bisa disuntik/ditarik per HARI
    max_storage_volume,          # MMBtu maksimum yang bisa disimpan sekaligus
    storage_cost_per_month,      # biaya sewa fasilitas ($) per bulan (flat fee)
    injection_cost_per_mmbtu=0.0,   # biaya injeksi ($/MMBtu)
    withdrawal_cost_per_mmbtu=0.0,  # biaya penarikan ($/MMBtu)
    transport_cost_per_event=0.0,   # biaya transportasi ($) per event (injeksi ATAU penarikan)
    price_lookup=None,           # fungsi date -> harga; default = estimate_price
    price_df=None,
    verbose=False,
):
    """
    Menghitung nilai (value) sebuah kontrak penyimpanan gas alam.

    Berikut ini adalah Parameter beserta penjelasannya:
    ---------
    injection_dates : list of (date, volume)
        Daftar tanggal & volume (MMBtu) gas yang DIBELI dan disuntikkan ke storage.
    withdrawal_dates : list of (date, volume)
        Daftar tanggal & volume (MMBtu) gas yang DITARIK dari storage dan DIJUAL.
    injection_withdrawal_rate : float
        Batas kecepatan operasi: MMBtu maksimum yang boleh disuntik ATAU ditarik
        pada satu tanggal/event.
    max_storage_volume : float
        Kapasitas maksimum fasilitas penyimpanan (MMBtu).
    storage_cost_per_month : float
        Biaya sewa flat fasilitas penyimpanan, dihitung berdasarkan lamanya gas
        disimpan (dari injeksi pertama sampai penarikan terakhir), dipro-rata
        harian lalu dikonversi ke basis bulanan (30 hari/bulan).
    injection_cost_per_mmbtu, withdrawal_cost_per_mmbtu : float
        Biaya operasional per MMBtu untuk setiap event injeksi/penarikan.
    transport_cost_per_event : float
        Biaya transportasi flat setiap kali ada perjalanan ke/dari fasilitas
        (dikenakan pada SETIAP event injeksi maupun penarikan).
    price_lookup : callable, optional
        Fungsi date -> harga $/MMBtu. Default memakai estimate_price() dari data
        historis Nat_Gas.csv.
    verbose : bool
        Jika True, mencetak rincian arus kas.

    Return
    ------
    dict berisi rincian arus kas dan 'contract_value' (nilai bersih kontrak, $).
    """

    if price_lookup is None:
        if price_df is None:
            price_df = load_price_data()
        price_lookup = lambda d: estimate_price(d, df=price_df)

    # Urutkan semua event dan cek validitas rate & kapasitas storage seiring waktu
    events = []
    for d, v in injection_dates:
        events.append((pd.to_datetime(d), "inject", v))
    for d, v in withdrawal_dates:
        events.append((pd.to_datetime(d), "withdraw", v))
    events.sort(key=lambda e: e[0])

    if not events:
        raise ValueError("Harus ada minimal satu tanggal injeksi dan satu tanggal penarikan.")

    current_volume = 0.0
    cash_flows = []
    total_buy_cost = 0.0
    total_sell_revenue = 0.0
    total_injection_withdrawal_cost = 0.0
    total_transport_cost = 0.0

    for date, action, volume in events:
        if volume <= 0:
            raise ValueError(f"Volume pada {date.date()} harus positif.")
        if volume > injection_withdrawal_rate:
            raise ValueError(
                f"Volume {volume:,.0f} MMBtu pada {date.date()} melebihi batas "
                f"kecepatan injeksi/penarikan sebesar {injection_withdrawal_rate:,.0f} MMBtu/hari."
            )

        price = price_lookup(date)

        if action == "inject":
            current_volume += volume
            if current_volume > max_storage_volume:
                raise ValueError(
                    f"Injeksi pada {date.date()} melebihi kapasitas maksimum "
                    f"penyimpanan ({max_storage_volume:,.0f} MMBtu). "
                    f"Volume tersimpan akan menjadi {current_volume:,.0f} MMBtu."
                )
            buy_cost = price * volume
            inj_cost = injection_cost_per_mmbtu * volume
            total_buy_cost += buy_cost
            total_injection_withdrawal_cost += inj_cost
            total_transport_cost += transport_cost_per_event
            cash_flows.append({
                "date": date.date(), "action": "INJECT (beli & simpan)",
                "volume_mmbtu": volume, "price_per_mmbtu": round(price, 4),
                "buy_cost": -round(buy_cost, 2),
                "injection_cost": -round(inj_cost, 2),
                "transport_cost": -round(transport_cost_per_event, 2),
            })

        else:  # withdraw
            current_volume -= volume
            if current_volume < -1e-6:
                raise ValueError(
                    f"Penarikan pada {date.date()} melebihi volume gas yang "
                    f"tersedia di storage."
                )
            sell_revenue = price * volume
            wd_cost = withdrawal_cost_per_mmbtu * volume
            total_sell_revenue += sell_revenue
            total_injection_withdrawal_cost += wd_cost
            total_transport_cost += transport_cost_per_event
            cash_flows.append({
                "date": date.date(), "action": "WITHDRAW (tarik & jual)",
                "volume_mmbtu": volume, "price_per_mmbtu": round(price, 4),
                "sell_revenue": round(sell_revenue, 2),
                "withdrawal_cost": -round(wd_cost, 2),
                "transport_cost": -round(transport_cost_per_event, 2),
            })

    # --- Biaya sewa penyimpanan (storage fee), dihitung dari injeksi pertama
    #     sampai penarikan terakhir, dipro-rata harian lalu dikonversi ke bulanan.
    first_date = events[0][0]
    last_date = events[-1][0]
    days_stored = max((last_date - first_date).days, 0)
    months_stored = days_stored / 30.0
    total_storage_cost = storage_cost_per_month * months_stored

    contract_value = (
        total_sell_revenue
        - total_buy_cost
        - total_storage_cost
        - total_injection_withdrawal_cost
        - total_transport_cost
    )

    result = {
        "cash_flows": cash_flows,
        "total_buy_cost": round(total_buy_cost, 2),
        "total_sell_revenue": round(total_sell_revenue, 2),
        "total_storage_cost": round(total_storage_cost, 2),
        "days_stored": days_stored,
        "total_injection_withdrawal_cost": round(total_injection_withdrawal_cost, 2),
        "total_transport_cost": round(total_transport_cost, 2),
        "contract_value": round(contract_value, 2),
    }

    if verbose:
        print(f"\n{'='*70}\nRINCIAN KONTRAK\n{'='*70}")
        for cf in cash_flows:
            print(cf)
        print(f"{'-'*70}")
        print(f"Total pendapatan penjualan   : ${result['total_sell_revenue']:,.2f}")
        print(f"Total biaya pembelian        : -${result['total_buy_cost']:,.2f}")
        print(f"Total biaya sewa storage     : -${result['total_storage_cost']:,.2f}  ({days_stored} hari ~ {months_stored:.2f} bulan)")
        print(f"Total biaya injeksi/tarik    : -${result['total_injection_withdrawal_cost']:,.2f}")
        print(f"Total biaya transportasi     : -${result['total_transport_cost']:,.2f}")
        print(f"{'-'*70}")
        print(f"NILAI KONTRAK (net value)    : ${result['contract_value']:,.2f}")
        print(f"{'='*70}\n")

    return result

## PENGUJIAN / CONTOH PENGGUNAAN

In [ ]:
if __name__ == "__main__":

    price_df = load_price_data("/content/drive/MyDrive/Forage/Nat_Gas.csv")

    # CONTOH 1: Mereplikasi contoh sederhana di penjelasan tugas
    #   Beli 1 juta MMBtu di musim panas @ $2, simpan 4 bulan, jual @ $3.
    #   Storage $100k/bulan, biaya injeksi/tarik total $10k (dibagi $5k inject +
    #   $5k withdraw), transportasi $50k setiap kali (2x = $100k).
    #   Nilai kontrak yang diharapkan: (3-2)*1e6 - 400,000 - 10,000 - 100,000 = $490,000

    def flat_price_lookup(date):
        date = pd.to_datetime(date)
        return 2.0 if date.month in (6, 7, 8) else 3.0  # musim panas vs musim dingin (ilustrasi)

    example1 = price_storage_contract(
        injection_dates=[("2024-06-01", 1_000_000)],
        withdrawal_dates=[("2024-10-01", 1_000_000)],
        injection_withdrawal_rate=1_000_000,
        max_storage_volume=1_000_000,
        storage_cost_per_month=100_000,
        injection_cost_per_mmbtu=0.005,   # $5,000 / 1,000,000 MMBtu
        withdrawal_cost_per_mmbtu=0.005,  # $5,000 / 1,000,000 MMBtu
        transport_cost_per_event=50_000,
        price_lookup=flat_price_lookup,
        verbose=True,
    )
    print(f"Nilai kontrak Contoh 1: ${example1['contract_value']:,.2f} "
          f"(ekspektasi ~$490,000)\n")

    # CONTOH 2: Kasus nyata klien -- beli gas SEKARANG (musim panas 2026, di luar
    # rentang data historis sehingga memakai model ekstrapolasi) untuk disimpan
    # dan dijual saat musim dingin karena klien memperkirakan musim dingin akan
    # lebih dingin (harga lebih tinggi) dari perkiraan pasar.

    example2 = price_storage_contract(
        injection_dates=[("2026-07-15", 500_000), ("2026-08-15", 500_000)],
        withdrawal_dates=[("2026-12-15", 600_000), ("2027-01-15", 400_000)],
        injection_withdrawal_rate=750_000,     # maks 750,000 MMBtu per hari
        max_storage_volume=1_000_000,          # kapasitas storage 1 juta MMBtu
        storage_cost_per_month=100_000,
        injection_cost_per_mmbtu=0.01,
        withdrawal_cost_per_mmbtu=0.01,
        transport_cost_per_event=50_000,
        price_df=price_df,                     # otomatis pakai estimate_price()
        verbose=True,
    )
    print(f"Nilai kontrak Contoh 2: ${example2['contract_value']:,.2f}\n")


    # CONTOH 3: Kontrak dengan beberapa siklus injeksi/penarikan dalam satu musim,
    # dan memakai injeksi lebih besar dari batas rate (untuk menunjukkan validasi
    # error yang bekerja).

    try:
        price_storage_contract(
            injection_dates=[("2026-06-01", 2_000_000)],  # melebihi rate 1,000,000/hari
            withdrawal_dates=[("2026-12-01", 2_000_000)],
            injection_withdrawal_rate=1_000_000,
            max_storage_volume=2_000_000,
            storage_cost_per_month=100_000,
            price_df=price_df,
        )
    except ValueError as e:
        print(f"Contoh 3 (validasi rate) berhasil menangkap error yang diharapkan:\n  -> {e}\n")


    # CONTOH 4: Estimasi harga pada beberapa tanggal (dalam & luar rentang data)

    print("Contoh estimasi harga pada beberapa tanggal:")
    for d in ["2021-03-15", "2023-11-20", "2024-09-30", "2025-01-15", "2026-12-15"]:
        print(f"  {d} -> ${estimate_price(d, df=price_df):.2f} /MMBtu")


RINCIAN KONTRAK
{'date': datetime.date(2024, 6, 1), 'action': 'INJECT (beli & simpan)', 'volume_mmbtu': 1000000, 'price_per_mmbtu': 2.0, 'buy_cost': -2000000.0, 'injection_cost': -5000.0, 'transport_cost': -50000}
{'date': datetime.date(2024, 10, 1), 'action': 'WITHDRAW (tarik & jual)', 'volume_mmbtu': 1000000, 'price_per_mmbtu': 3.0, 'sell_revenue': 3000000.0, 'withdrawal_cost': -5000.0, 'transport_cost': -50000}
----------------------------------------------------------------------
Total pendapatan penjualan   : $3,000,000.00
Total biaya pembelian        : -$2,000,000.00
Total biaya sewa storage     : -$406,666.67  (122 hari ~ 4.07 bulan)
Total biaya injeksi/tarik    : -$10,000.00
Total biaya transportasi     : -$100,000.00
----------------------------------------------------------------------
NILAI KONTRAK (net value)    : $483,333.33

Nilai kontrak Contoh 1: $483,333.33 (ekspektasi ~$490,000)


RINCIAN KONTRAK
{'date': datetime.date(2026, 7, 15), 'action': 'INJECT (beli & simpan)'